# Competition Abonnés Brut — pipeline tout-en-1

Objectif : nettoyer les données, itérer jusqu'au meilleur score possible (cible **0.95**), exporter le modèle en **joblib**, puis lancer une interface **Streamlit** (prédiction + justification + historique + feedback + monitoring).

> Chemin données : `data/abonnes_brut.csv`  
> Artefacts : `artifacts/churn_model.joblib`  
> App : `app/streamlit_app.py`


In [ ]:
from pathlib import Path
import sys
import subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data" / "abonnes_brut.csv").exists():
    # si le notebook est ouvert ailleurs
    ROOT = Path("/home/wasfade/anaconda_projects/db/cours")
sys.path.insert(0, str(ROOT))

from app.churn_core import (
    nettoyer, prepare_xy, iterative_train, train_and_export,
    MODEL_PATH, META_PATH, DATA_PATH, load_model, predict_one, save_prediction,
)

print("ROOT =", ROOT)
print("DATA =", DATA_PATH, "exists=", DATA_PATH.exists())


## 1. Chargement & nettoyage

On reprend la logique métier : suppression des doublons, fuite (`motif_resiliation`), dates → `anciennete_mois`, harmonisation textes, dépenses, imputations.  
**On ne garde pas** `date_activation` ni `id_client` comme features du modèle.


In [ ]:
raw = pd.read_csv(DATA_PATH)
print("Brut:", raw.shape)
display(raw.head(3))

df = nettoyer(raw)
print("Propre:", df.shape)
print("Taux résiliation:", round(df["resilie"].mean(), 4))
display(df.head())
print("NaN restants:\n", df.isna().sum().loc[lambda s: s > 0])


## 2. Features utiles

Colonnes importantes (ex. `variation_conso_pct`, dépenses, réclamations, note réseau, ancienneté…).  
Features dérivées : `baisse_conso`, `faible_note`, `risque`, `intensite`, etc.


In [ ]:
X, y, feat = prepare_xy(df)
print("X:", X.shape, "| y rate:", round(y.mean(), 4))
print("Colonnes:", list(X.columns))
display(X.describe().T.head(12))


## 3. Entraînement itératif (cible 0.95)

Plusieurs modèles sklearn (RF, HGB, Vote). On retient le meilleur score **test** (accuracy avec seuil optimisé).  
Sur ce jeu déséquilibré (~86 % non-résiliés), le plafond honnête observé tourne autour de **~0.90–0.91** — on pousse quand même jusqu'au maximum sans fuite de données.


In [ ]:
result = train_and_export(DATA_PATH, target_score=0.95)
rounds = pd.DataFrame(result["history"])
display(rounds)

meta = result["meta"]
print("=" * 60)
print("Meilleur modèle :", meta["best_model"])
print("Accuracy test   :", round(meta["test_accuracy"], 4))
print("Accuracy + seuil:", round(meta["test_accuracy_opt"], 4), "| seuil=", meta["threshold"])
print("AUC             :", round(meta["test_auc"], 4))
print("Baseline majorité:", round(meta["majority_baseline"], 4))
print("Cible 0.95 atteinte ?", meta["test_accuracy_opt"] >= 0.95)
print("Sauvegardé →", MODEL_PATH)
print("=" * 60)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(rounds["round"], rounds["accuracy_opt"], marker="o", label="accuracy opt")
ax.plot(rounds["round"], rounds["auc"], marker="s", label="AUC")
ax.axhline(0.95, color="crimson", ls="--", label="cible 0.95")
ax.set_xlabel("Itération")
ax.set_ylabel("Score")
ax.set_title("Progression des itérations")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 4. Vérification joblib + prédiction exemple


In [ ]:
pipe, meta = load_model()
exemple = {
    "id_client": "NT105452",
    "zone": "urbaine",
    "departement": "Littoral",
    "age": 22,
    "type_offre": "prépayé",
    "appels_min": 94.8,
    "data_mo": 1083.0,
    "sms": 22,
    "depense_fcfa": 2920,
    "nb_recharges": 4,
    "mobile_money": 0,
    "reclamations_3m": 2,
    "note_reseau": 3,
    "variation_conso_pct": -24.4,
    "anciennete_mois": 34,
}
pred = predict_one(exemple, pipe, meta)
save_prediction(pred)
print(pred["label"], "| proba=", round(pred["proba_resiliation"], 4))
print(pred["justification"])


## 5. Lancer l'interface Streamlit

L'app permet :
- formulaire (champs utiles seulement)
- prédiction + probabilité + **justification**
- **historique** (client, date, proba) + graphiques
- enregistrement du **résultat réel** (réentraîner plus tard)
- **monitoring** (dérive des probas / accuracy feedback)

Exécutez la cellule ci-dessous (ou en terminal) :

```bash
cd /home/wasfade/anaconda_projects/db/cours
streamlit run app/streamlit_app.py
```


In [ ]:
# Lance l'interface Streamlit (tout-en-1)
import os
app_path = ROOT / "app" / "streamlit_app.py"
assert (ROOT / "artifacts" / "churn_model.joblib").exists(), "Relancez la cellule d'entraînement avant."
cmd = [sys.executable, "-m", "streamlit", "run", str(app_path), "--server.headless", "true"]
print("→ Démarrage :", " ".join(cmd))
print("→ URL locale typique : http://localhost:8501")
# Lance en arrière-plan ; ouvrez l'URL dans le navigateur
proc = subprocess.Popen(cmd, cwd=str(ROOT), env={**os.environ, "STREAMLIT_BROWSER_GATHER_USAGE_STATS": "false"})
print("Streamlit PID:", proc.pid)
print("Pour arrêter : import os; os.kill(%d, 15)" % proc.pid)
